# Simulating a Tool-Using AI Agent for Logistics Management

**Copyright (c) 2026 Shrikara Kaudambady. All rights reserved.**

This notebook demonstrates an AI agent designed for a common logistics task: **smart order fulfillment**. The agent's goal is to decide which warehouse should ship a new customer order. To do this, it will use a set of 'tools' to check inventory, calculate shipping costs, and estimate delivery times, then make a recommendation based on a specific objective (e.g., lowest cost or fastest delivery).

### 1. Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np

### 2. Simulate the Retail Environment

First, we create a simulated environment for our online retailer. This includes warehouses with inventory, product information, and data models for shipping costs and times. In a real system, this data would come from databases and APIs.

In [ ]:
# Warehouse Data with Stock Levels
warehouses = pd.DataFrame([
    {'warehouse_id': 'WH-CA', 'city': 'Los Angeles', 'stock_levels': {'P101': 100, 'P102': 50, 'P103': 0}},
    {'warehouse_id': 'WH-NY', 'city': 'New York', 'stock_levels': {'P101': 80, 'P102': 10, 'P103': 200}},
    {'warehouse_id': 'WH-TX', 'city': 'Dallas', 'stock_levels': {'P101': 120, 'P102': 0, 'P103': 150}}
]).set_index('warehouse_id')

# Product Data
products = pd.DataFrame([
    {'product_id': 'P101', 'name': 'Standard Widget', 'weight_kg': 1.5},
    {'product_id': 'P102', 'name': 'Heavy Widget', 'weight_kg': 5.0},
    {'product_id': 'P103', 'name': 'Premium Widget', 'weight_kg': 2.0}
]).set_index('product_id')

# Shipping Costs ($ per kg to different regions)
shipping_costs = {
    'WH-CA': {'West': 1.50, 'Central': 2.50, 'East': 3.50},
    'WH-NY': {'West': 3.50, 'Central': 2.50, 'East': 1.50},
    'WH-TX': {'West': 2.00, 'Central': 1.00, 'East': 2.00}
}

# Delivery Times (in days)
delivery_times = {
    'WH-CA': {'West': 2, 'Central': 4, 'East': 6},
    'WH-NY': {'West': 6, 'Central': 4, 'East': 2},
    'WH-TX': {'West': 3, 'Central': 2, 'East': 3}
}

print("Environment simulation is ready.")

### 3. Define the AI Agent and its Tools

We'll create a `FulfillmentAgent` class. The agent's intelligence comes from its ability to use its tools (`check_stock`, `get_shipping_cost`, etc.) to gather information and then reason over that information to meet a goal.

In [ ]:
class FulfillmentAgent:
    def __init__(self, warehouses_df, products_df, costs, times):
        # The agent gets access to the environment's data
        self.warehouses = warehouses_df
        self.products = products_df
        self.costs = costs
        self.times = times
        print("Fulfillment Agent Initialized.")
    
    # --- AGENT'S TOOLS ---
    def check_stock(self, warehouse_id, product_id, quantity):
        stock = self.warehouses.loc[warehouse_id, 'stock_levels'].get(product_id, 0)
        return stock >= quantity

    def get_shipping_cost(self, from_warehouse, to_region, product_id, quantity):
        weight = self.products.loc[product_id, 'weight_kg']
        total_weight = weight * quantity
        cost_per_kg = self.costs[from_warehouse][to_region]
        return total_weight * cost_per_kg
        
    def get_delivery_time(self, from_warehouse, to_region):
        return self.times[from_warehouse][to_region]
    
    # --- AGENT'S REASONING LOGIC ---
    def recommend_warehouse(self, order, objective='lowest_cost'):
        print(f"\n=========================================================")
        print(f"Received new order: {order['quantity']}x {order['product_id']} to {order['destination_region']}.")
        print(f"Objective: {objective}.")
        print("---------------------------------------------------------")

        # Step 1: Find all warehouses with enough stock (using the 'check_stock' tool)
        print("\n1. Agent is checking stock levels...")
        valid_warehouses = []
        for wh_id in self.warehouses.index:
            if self.check_stock(wh_id, order['product_id'], order['quantity']):
                valid_warehouses.append(wh_id)
        
        if not valid_warehouses:
            print("\nRESULT: No warehouse has sufficient stock. Fulfillment failed.")
            print("=========================================================")
            return None
        
        print(f"   -> Found {len(valid_warehouses)} valid options: {valid_warehouses}")
        
        # Step 2: Gather data for each valid option (using cost and time tools)
        print("\n2. Agent is gathering cost and delivery data for each option...")
        options = []
        for wh_id in valid_warehouses:
            cost = self.get_shipping_cost(wh_id, order['destination_region'], order['product_id'], order['quantity'])
            time = self.get_delivery_time(wh_id, order['destination_region'])
            options.append({'warehouse_id': wh_id, 'shipping_cost': cost, 'delivery_days': time})
        
        options_df = pd.DataFrame(options).set_index('warehouse_id')
        print("   -> Data gathered:")
        print(options_df)
        
        # Step 3: Evaluate options based on the objective
        print(f"\n3. Agent is evaluating options based on '{objective}'...")
        if objective == 'lowest_cost':
            best_option_id = options_df['shipping_cost'].idxmin()
        elif objective == 'fastest_delivery':
            best_option_id = options_df['delivery_days'].idxmin()
        else:
            print("   -> Unknown objective. Defaulting to lowest cost.")
            best_option_id = options_df['shipping_cost'].idxmin()
        
        best_option_details = options_df.loc[best_option_id]
        print(f"   -> Best option found: {best_option_id}")
        
        # Step 4: Make final recommendation
        print("\n*** AGENT'S FINAL RECOMMENDATION ***")
        print(f"Ship order from: {best_option_id}")
        print(f"- Shipping Cost: ${best_option_details['shipping_cost']:.2f}")
        print(f"- Estimated Delivery: {best_option_details['delivery_days']} days")
        print("=========================================================")
        return best_option_details.to_dict()


### 4. Demonstrate the Agent in Action

Let's create a new order and see how the agent handles it with different objectives.

In [ ]:
# Initialize the agent with our simulated environment
agent = FulfillmentAgent(warehouses, products, shipping_costs, delivery_times)

# Define a new customer order
new_order = {
    'product_id': 'P101',
    'quantity': 5,
    'destination_region': 'East'
}

# --- SCENARIO 1: Find the CHEAPEST fulfillment option ---
agent.recommend_warehouse(new_order, objective='lowest_cost')

# --- SCENARIO 2: Find the FASTEST fulfillment option ---
agent.recommend_warehouse(new_order, objective='fastest_delivery')

# --- SCENARIO 3: Handle an out-of-stock item ---
out_of_stock_order = {
    'product_id': 'P102',
    'quantity': 20,
    'destination_region': 'West'   
}
agent.recommend_warehouse(out_of_stock_order, objective='lowest_cost')